# DENTEX Curriculum Training — Google Colab

Picks up from a saved Stage 2 warm-start checkpoint and runs:
- **Stage 2** — Tooth Enumeration (warm-start from `best.pt`)
- **Stage 3** — Diagnosis + Segmentation (curriculum, warm-start from Stage 2)
- **Baseline** — Diagnosis + Segmentation (ImageNet init, no curriculum)

All data and checkpoints are read from / written to Google Drive.

**Drive layout expected:**
```
My Drive/6600-project-data/
    stage2_enumeration/
    stage3_disease/
    yamls/
    best.pt          <- Stage 2 warm-start checkpoint
```

## 0. Runtime check

Before running anything: **Runtime → Change runtime type → T4 GPU**. Confirm below.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected. Go to Runtime → Change runtime type → T4 GPU and re-run."
    )

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU   : {gpu}")
print(f"VRAM  : {vram:.1f} GB")
print(f"CUDA  : {torch.version.cuda}")
print(f"Torch : {torch.__version__}")

## 1. Install dependencies

In [ ]:
%pip install -q ultralytics
import ultralytics
print(f"ultralytics {ultralytics.__version__}")

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Paths

In [ ]:
from pathlib import Path
import shutil, os

# ── Drive root ────────────────────────────────────────────────────────────────
DRIVE_ROOT   = Path("/content/drive/MyDrive/6600-project-data")

# ── Data (read from Drive) ────────────────────────────────────────────────────
YAMLS_DIR    = DRIVE_ROOT / "yamls"
STAGE2_YAML  = YAMLS_DIR  / "stage2_enumeration.yaml"
STAGE3_YAML  = YAMLS_DIR  / "stage3_disease.yaml"

# ── Checkpoints (written to Drive so they persist after session ends) ─────────
CKPT_DIR     = DRIVE_ROOT / "checkpoints"
CKPT_DIR.mkdir(exist_ok=True)

STAGE2_WARMSTART = DRIVE_ROOT / "best.pt"          # interrupted local run
STAGE2_BEST      = CKPT_DIR   / "stage2_best.pt"
STAGE3_BEST      = CKPT_DIR   / "stage3_best.pt"
BASELINE_BEST    = CKPT_DIR   / "baseline_best.pt"

# ── Local Colab workspace (fast SSD, used for run artifacts) ──────────────────
RUNS_DIR     = Path("/content/runs")
RUNS_DIR.mkdir(exist_ok=True)

# ── Verify everything is in place ─────────────────────────────────────────────
checks = {
    "stage2_enumeration.yaml" : STAGE2_YAML,
    "stage3_disease.yaml"     : STAGE3_YAML,
    "best.pt (warm-start)"    : STAGE2_WARMSTART,
}
all_ok = True
for label, p in checks.items():
    status = "✓" if p.exists() else "✗ MISSING"
    print(f"  {status}  {label}")
    if not p.exists():
        all_ok = False

if not all_ok:
    raise RuntimeError("One or more required files are missing from Drive. Check the layout above.")

print("\nAll files found. Ready to train.")

## 4. Patch YAML paths

The YAMLs were generated on your local machine and contain absolute local paths.
This cell rewrites the `path:` field in each YAML to point to the Drive location.

In [ ]:
import re

def patch_yaml_path(yaml_path: Path, new_data_root: Path) -> None:
    """Replace the `path:` line in a YOLO YAML with the correct Colab path."""
    text = yaml_path.read_text()
    text = re.sub(r'^path:.*$', f'path: {new_data_root}', text, flags=re.MULTILINE)
    yaml_path.write_text(text)
    print(f"  Patched: {yaml_path.name}  →  path: {new_data_root}")

patch_yaml_path(STAGE2_YAML, DRIVE_ROOT / "stage2_enumeration")
patch_yaml_path(STAGE3_YAML, DRIVE_ROOT / "stage3_disease")

print("\nYAML paths updated.")

## 5. Hyperparameters

In [ ]:
DEVICE   = "cuda"
IMGSZ    = 640
BATCH    = 16     # T4 has 15 GB VRAM — 16 is safe; bump to 32 on A100

EPOCHS_S2  = 100
EPOCHS_S3  = 100
EPOCHS_BL  = 100

PATIENCE_S2 = 20
PATIENCE_S3 = 25
PATIENCE_BL = 25

print(f"Device : {DEVICE}")
print(f"Batch  : {BATCH}")
print(f"ImgSz  : {IMGSZ}")

## 6. Training helper

In [ ]:
from ultralytics import YOLO

def run_stage(
    name, model_path, yaml_path, run_name,
    epochs, lr0, patience, copy_paste=0.0,
    hsv_s=0.4, hsv_v=0.4,
    warmup_epochs=3,
    checkpoint_dst=None,
):
    """Train one stage, copy best.pt to Drive, return (ckpt_path, results)."""
    print(f"\n{'='*60}")
    print(f"  {name}")
    print(f"  Warm-start : {model_path}")
    print(f"  YAML       : {yaml_path}")
    print(f"  Epochs     : {epochs}  |  Patience : {patience}")
    print(f"{'='*60}\n")

    model   = YOLO(str(model_path))
    results = model.train(
        data           = str(yaml_path),
        epochs         = epochs,
        imgsz          = IMGSZ,
        batch          = BATCH,
        device         = DEVICE,
        project        = str(RUNS_DIR),
        name           = run_name,
        exist_ok       = True,
        optimizer      = "AdamW",
        lr0            = lr0,
        lrf            = 0.01,
        warmup_epochs  = warmup_epochs,
        weight_decay   = 0.0005,
        augment        = True,
        hsv_h          = 0.015,
        hsv_s          = hsv_s,
        hsv_v          = hsv_v,
        flipud         = 0.1,
        fliplr         = 0.5,
        mosaic         = 1.0,
        copy_paste     = copy_paste,
        patience       = patience,
        save           = True,
        plots          = True,
        verbose        = True,
    )

    # Copy best checkpoint to Drive immediately so it survives session expiry
    best_src = Path(results.save_dir) / "weights" / "best.pt"
    if best_src.exists() and checkpoint_dst is not None:
        shutil.copy(best_src, checkpoint_dst)
        print(f"\n✓ Checkpoint saved to Drive → {checkpoint_dst}")
    else:
        print(f"\n⚠ best.pt not found at {best_src}")

    return best_src, results

## 7. Stage 2 — Tooth Enumeration

Warm-starts from the interrupted local `best.pt` checkpoint.
Continues fine-tuning the 8-class tooth position task.

In [ ]:
s2_ckpt, s2_results = run_stage(
    name           = "Stage 2 — Tooth Enumeration",
    model_path     = STAGE2_WARMSTART,
    yaml_path      = STAGE2_YAML,
    run_name       = "stage2",
    epochs         = EPOCHS_S2,
    lr0            = 0.0005,
    patience       = PATIENCE_S2,
    warmup_epochs  = 2,
    hsv_s          = 0.3,
    hsv_v          = 0.3,
    checkpoint_dst = STAGE2_BEST,
)

## 8. Stage 3 — Diagnosis Detection (Curriculum)

Fine-tunes from Stage 2 checkpoint on the 4-class disease task with segmentation masks.
This is the final curriculum model.

In [ ]:
s3_ckpt, s3_results = run_stage(
    name           = "Stage 3 — Diagnosis Detection (Curriculum)",
    model_path     = STAGE2_BEST,
    yaml_path      = STAGE3_YAML,
    run_name       = "stage3_curriculum",
    epochs         = EPOCHS_S3,
    lr0            = 0.0003,
    patience       = PATIENCE_S3,
    warmup_epochs  = 2,
    copy_paste     = 0.1,
    checkpoint_dst = STAGE3_BEST,
)

## 9. Baseline — Single-Stage (No Curriculum)

Trains from ImageNet pretrained weights directly on the disease data.
Same hyperparameters as Stage 3 for a fair RQ1 comparison.

In [ ]:
bl_ckpt, bl_results = run_stage(
    name           = "Baseline — Single-Stage (No Curriculum)",
    model_path     = "yolov8m-seg.pt",
    yaml_path      = STAGE3_YAML,
    run_name       = "baseline",
    epochs         = EPOCHS_BL,
    lr0            = 0.001,
    patience       = PATIENCE_BL,
    copy_paste     = 0.1,
    checkpoint_dst = BASELINE_BEST,
)

## 10. Save run artifacts to Drive

Zips the full `runs/` directory (training curves, confusion matrices, etc.)
and copies it to Drive so you can pull it back to your local machine for
analysis in `02_evaluation.ipynb`.

In [ ]:
import shutil
from pathlib import Path

ARTIFACTS_ZIP = DRIVE_ROOT / "colab_runs.zip"

print("Zipping run artifacts...")
shutil.make_archive(
    base_name = str(DRIVE_ROOT / "colab_runs"),
    format    = "zip",
    root_dir  = str(RUNS_DIR.parent),
    base_dir  = RUNS_DIR.name,
)
print(f"✓ Artifacts saved → {ARTIFACTS_ZIP}")

# Summary of what was saved to Drive
print("\n── Checkpoints saved to Drive ──────────────────────")
for label, p in [
    ("Stage 2 best",  STAGE2_BEST),
    ("Stage 3 best",  STAGE3_BEST),
    ("Baseline best", BASELINE_BEST),
]:
    if p.exists():
        size_mb = p.stat().st_size / 1e6
        print(f"  ✓  {label:<15} {p.name}  ({size_mb:.1f} MB)")
    else:
        print(f"  ✗  {label:<15} NOT FOUND")

print("\nDone. Download the checkpoints and colab_runs.zip to your local machine,")
print("then place them in models/checkpoints/ and run 02_evaluation.ipynb.")

## 11. Quick training curve preview

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

runs = {
    "Stage 3 (Curriculum)" : RUNS_DIR / "stage3_curriculum",
    "Baseline"             : RUNS_DIR / "baseline",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (label, run_dir) in zip(axes, runs.items()):
    results_png = run_dir / "results.png"
    if results_png.exists():
        img = mpimg.imread(results_png)
        ax.imshow(img)
        ax.set_title(label, fontsize=12)
    else:
        ax.text(0.5, 0.5, f"Not available\n{run_dir.name}",
                ha="center", va="center", transform=ax.transAxes)
    ax.axis("off")

plt.suptitle("Training Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()